In [0]:
import subprocess

from pyspark.sql import functions as F
from pyspark.sql.types import (
    ArrayType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)


command_result_schema = ArrayType(
    StructType([
        StructField("command", StringType(), nullable=False),
        StructField("exit_code", IntegerType(), nullable=True),
        StructField("output", StringType(), nullable=True),
    ])
)


@F.udf(returnType=command_result_schema)
def run_commands_udf(commands: list[str]):
    results = []

    for command in commands:
        try:
            process = subprocess.run(
                command,
                shell=True,
                capture_output=True,
                text=True,
                timeout=10,
            )

            output = (
                process.stdout.strip()
                or process.stderr.strip()
                or ""
            )

            results.append({
                "command": command,
                "exit_code": process.returncode,
                "output": output,
            })

        except Exception as exc:
            results.append({
                "command": command,
                "exit_code": None,
                "output": f"{type(exc).__name__}: {exc}",
            })

    return results

def run_worker_commands(commands: list[str]):
    commands_column = F.array(
        *[F.lit(command) for command in commands]
    )

    return (
        spark.range(1)
        .select(
            F.explode(
                run_commands_udf(commands_column)
            ).alias("result")
        )
        .select(
            F.col("result.command").alias("command"),
            F.col("result.exit_code").alias("exit_code"),
            F.col("result.output").alias("output"),
        )
    )  